02 - Preprocessing: Language Detection, Sentence Segmentation, Normalization

## Setup
Run the cell below first. It detects whether you're in **Google Colab** or
running **locally in VS Code**, and gets the environment ready either way
(clones the repo in Colab, installs requirements, downloads NLTK data, and
adds `src/` to the path so `pipeline.py` can be imported).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import csv
import pandas as pd

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv')

In [5]:
df.head()

,label,text
0,1,Congratulations! You've been selected for a lu...
1,1,URGENT: Your account has been compromised. Cli...
2,1,You've won a free iPhone! Claim your prize by ...
3,1,Act now and receive a 50% discount on all purc...
4,1,Important notice: Your subscription will expir...


In [6]:
df2 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

In [7]:
df2.head()

,Email Text,Email Type
0,"Dear Jordan, your subscription has been succes...",Safe Email
1,"Dear Casey, thank you for your purchase. Your ...",Safe Email
2,Congratulations! You've won a $3000 gift card....,Phishing Email
3,You have a new secure message from your bank. ...,Phishing Email
4,Your package delivery is pending. Please provi...,Phishing Email


In [8]:
# ============================================================
# SETUP CELL - run this first, every time
# Works both locally (VS Code / Jupyter) and in Google Colab
# ============================================================
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(f"{REPO_DIR}/notebooks")

    get_ipython().system("pip install -q -r ../requirements.txt")

    # Data files are large - if they were not committed to the repo,
    # upload them here once per Colab session.
    if not os.path.exists("../data/data.csv"):
        print("data/data.csv not found in the cloned repo.")
        print("Option A: git add + commit + push the CSVs from your")
        print("          local machine so they come down with the clone.")
        print("Option B: uncomment the lines below to upload manually.")
        # from google.colab import files
        # uploaded = files.upload()   # select data.csv + validation_dataset.csv
        # os.makedirs("../data", exist_ok=True)
        # for fname in uploaded:
        #     os.rename(fname, f"../data/{fname}")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

sys.path.append(os.path.abspath("../src"))
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())


data/data.csv not found in the cloned repo.
Option A: git add + commit + push the CSVs from your
          local machine so they come down with the clone.
Option B: uncomment the lines below to upload manually.
IN_COLAB = True
Working directory: /content/NLP_Ctrl-Alt-Elite/notebooks


### Step 1 - Language Detection
Non-English emails cannot be reliably classified by an English-trained
model, so we detect language and keep only English content.

In [9]:
get_ipython().system('pip install langdetect -qq')
from langdetect import detect, LangDetectException
import pandas as pd

# df_train = pd.read_csv('../data/train_renamed.csv') # Original line
df_train = df.rename(columns={'text': 'email_text', 'label': 'email_label'}) # Corrected: Use existing df and rename columns

def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

print('Detecting languages... this can take 1-2 minutes on 55k rows')
df_train['language'] = df_train['email_text'].apply(detect_language)

print(df_train['language'].value_counts().head(10))
en_pct = (df_train['language'] == 'en').mean() * 100
print(f'English emails: {en_pct:.1f}%')

df_en = df_train[df_train['language'] == 'en'].copy()
print(f'After filter: {len(df_en)} emails remaining (was {len(df_train)})')
print(df_en['email_label'].value_counts())

df_en.to_csv('../data/filtered_english.csv', index=False)
print('Saved: filtered_english.csv')

Detecting languages... this can take 1-2 minutes on 55k rows
language
en    41847
ro      552
ca      521
de      292
af      241
es      192
it      173
fr      171
nl      148
so      135
Name: count, dtype: int64
English emails: 92.7%
After filter: 41847 emails remaining (was 45155)
email_label
0    20951
1    20896
Name: count, dtype: int64
Saved: filtered_english.csv


### Step 2 - Sentence Segmentation
Spam tends to use short, punchy sentences ("Click here!", "Limited time
offer!"). Segmenting reveals this structural signal.

In [11]:
import nltk
from nltk.tokenize import sent_tokenize
import os # Import os for path manipulation
import pandas as pd # Ensure pandas is imported as it's needed for pd.read_csv

# Define REPO_ROOT_ABS_PATH as it's needed in this cell
REPO_ROOT_ABS_PATH = os.path.join("/content", "NLP_Ctrl-Alt-Elite")

# Construct the absolute path to the input data file
filtered_english_csv_path = os.path.join(REPO_ROOT_ABS_PATH, 'data', 'filtered_english.csv')
df = pd.read_csv(filtered_english_csv_path)

def segment_sentences(text):
    return sent_tokenize(str(text))

def count_sentences(text):
    return len(sent_tokenize(str(text)))

def avg_sentence_length(text):
    sentences = sent_tokenize(str(text))
    if not sentences:
        return 0
    return sum(len(s.split()) for s in sentences) / len(sentences)

df['sentence_count'] = df['email_text'].apply(count_sentences)
df['avg_sent_len'] = df['email_text'].apply(avg_sentence_length)

print('Sentence count comparison:')
print(df.groupby('email_label')['sentence_count'].describe())
print()
print('Average sentence length comparison:')
print(df.groupby('email_label')['avg_sent_len'].mean())

# Construct the absolute path for saving the segmented file
segmented_csv_path = os.path.join(REPO_ROOT_ABS_PATH, 'data', 'segmented.csv')
df.to_csv(segmented_csv_path, index=False)
print('Saved:', segmented_csv_path)


Sentence count comparison:
               count      mean        std  min  25%  50%  75%     max
email_label                                                          
0            20951.0  7.295165  21.556542  1.0  1.0  2.0  7.0  1565.0
1            20896.0  3.500957  12.618556  1.0  1.0  1.0  1.0   693.0

Average sentence length comparison:
email_label
0     13.487005
1    139.835770
Name: avg_sent_len, dtype: float64
Saved: /content/NLP_Ctrl-Alt-Elite/data/segmented.csv


### Step 3 - Text Normalization
Spam uses abbreviations ("ur", "u", "gr8"), ALL CAPS, and symbols to
dodge filters. Normalizing standardizes vocabulary for the model. The
reusable functions live in `src/pipeline.py` so later notebooks (and the
group's final web app) share the exact same logic.

In [12]:
import sys, os
import pandas as pd # Ensure pandas is imported

print(f"Current working directory: {os.getcwd()}")

# The repository 'NLP_Ctrl-Alt-Elite' is cloned into /content/NLP_Ctrl-Alt-Elite by the setup cell (37b26596).
# Construct the absolute root path of the repository.
REPO_ROOT_ABS_PATH = os.path.join("/content", "NLP_Ctrl-Alt-Elite")

# Construct the absolute path to the 'src' directory, including the nested folder 'cit-24-01-0182'
# as 'pipeline.py' appears to be inside this sub-directory.
path_to_add = os.path.join(REPO_ROOT_ABS_PATH, "src", "cit-24-01-0182")

print(f"Path being added to sys.path: {path_to_add}")
sys.path.insert(0, path_to_add) # Use insert(0) to give it highest priority

# Verify the existence of pipeline.py
pipeline_module_path = os.path.join(path_to_add, 'pipeline.py')
if os.path.exists(pipeline_module_path):
    print(f"Verification: pipeline.py found at {pipeline_module_path}")
else:
    print(f"Verification: pipeline.py NOT found at {pipeline_module_path}")
    print(f"Contents of directory {path_to_add}: {os.listdir(path_to_add) if os.path.exists(path_to_add) else 'Directory does not exist'}\n")

# Import functions from pipeline
from pipeline import normalize_text, tokenize_and_lemmatize

# Construct the absolute path to the data file
segmented_csv_path = os.path.join(REPO_ROOT_ABS_PATH, 'data', 'segmented.csv')
df = pd.read_csv(segmented_csv_path)

df['normalized_text'] = df['email_text'].apply(normalize_text)

print('BEFORE:', df['email_text'].iloc[0][:200])
print()
print('AFTER: ', df['normalized_text'].iloc[0][:200])

# Construct the absolute path for saving the preprocessed file
preprocessed_csv_path = os.path.join(REPO_ROOT_ABS_PATH, 'data', 'preprocessed.csv')
df.to_csv(preprocessed_csv_path, index=False)
print('Saved:', preprocessed_csv_path)

Current working directory: /content/NLP_Ctrl-Alt-Elite/notebooks
Path being added to sys.path: /content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182
Verification: pipeline.py found at /content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182/pipeline.py
BEFORE: Congratulations! You've been selected for a luxury vacation getaway. Claim your prize now!

AFTER:  congratulations you ve been selected for a luxury vacation getaway claim your prize now
Saved: /content/NLP_Ctrl-Alt-Elite/data/preprocessed.csv


### Also preprocess the validation dataset the same way

In [13]:
import os # Ensure os is imported here if it wasn't already

# The repository 'NLP_Ctrl-Alt-Elite' is cloned into /content/NLP_Ctrl-Alt-Elite by the setup cell.
REPO_ROOT_ABS_PATH = os.path.join("/content", "NLP_Ctrl-Alt-Elite")

# Use df2 which was loaded from validation_dataset.csv in a previous cell
# Rename columns of df2 to match expected format for language detection and normalization
# The original column names in df2 are 'Email Text' and 'Email Type'.
df_val = df2.rename(columns={'Email Text': 'email_text', 'Email Type': 'email_label'})

df_val['language'] = df_val['email_text'].apply(detect_language)
df_val = df_val[df_val['language'] == 'en'].copy()

df_val['normalized_text'] = df_val['email_text'].apply(normalize_text)

# Construct the absolute path for saving the preprocessed validation file
validation_preprocessed_csv_path = os.path.join(REPO_ROOT_ABS_PATH, 'data', 'validation_preprocessed.csv')
df_val.to_csv(validation_preprocessed_csv_path, index=False)
print(f'Validation set ready: {len(df_val)} records')

Validation set ready: 2000 records
